# Homework 6 — Tree Ensembles

ML Zoomcamp 2026 | Car fuel efficiency regression

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

DATA_PATH = Path('data/car_fuel_efficiency_2026.csv')

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def original_column(name):
    return name.split('=', 1)[0] if '=' in name else name

In [2]:
df = pd.read_csv(DATA_PATH)
target = 'fuel_efficiency_mpg'
feature_cols = [c for c in df.columns if c != target]
cat_cols = df[feature_cols].select_dtypes(include=['object', 'string']).columns.tolist()
for col in feature_cols:
    if col in cat_cols:
        df[col] = df[col].fillna('NA')  # no missing categoricals in this dataset
    else:
        df[col] = df[col].fillna(0.0)
print('Missing after fill:', df.isna().sum().sum())
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)
train_dicts = df_train[feature_cols].to_dict(orient='records')
val_dicts = df_val[feature_cols].to_dict(orient='records')
y_train = df_train[target].values
y_val = df_val[target].values
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
feature_names = dv.get_feature_names_out()

Missing after fill: 0


In [3]:
dt1 = DecisionTreeRegressor(max_depth=1)
dt1.fit(X_train, y_train)
root = feature_names[dt1.tree_.feature[0]]
print('Q1 root:', original_column(root))

Q1 root: model_year


In [4]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
print(f'Q2 RMSE: {rmse(y_val, rf.predict(X_val)):.4f}')

Q2 RMSE: 1.8332


In [5]:
for n in [10, 50, 100, 150]:
    m = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    m.fit(X_train, y_train)
    print(n, round(rmse(y_val, m.predict(X_val)), 3))

10 1.833


50 1.774


100 1.77


150 1.769


In [6]:
for d in [10, 15, 20, 25]:
    scores = []
    for n in [10, 50, 100, 150]:
        m = RandomForestRegressor(n_estimators=n, max_depth=d, random_state=1, n_jobs=-1)
        m.fit(X_train, y_train)
        scores.append(rmse(y_val, m.predict(X_val)))
    print('depth', d, 'mean', round(float(np.mean(scores)), 4))

depth 10 mean 1.7544


depth 15 mean 1.7842


depth 20 mean 1.7868


depth 25 mean 1.7863


In [7]:
m5 = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
m5.fit(X_train, y_train)
cols = ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']
agg = {c: 0.0 for c in cols}
for name, val in zip(feature_names, m5.feature_importances_):
    base = original_column(name)
    if base in agg:
        agg[base] += val
print(agg)
print('Q5:', max(agg, key=agg.get))

{'vehicle_weight': np.float64(0.20331343574711913), 'horsepower': np.float64(0.07827124720421208), 'acceleration': np.float64(0.054997632153652054), 'engine_displacement': np.float64(0.06128144481858746)}
Q5: vehicle_weight


In [8]:
X_tr = X_train.toarray(); X_va = X_val.toarray()
dtrain = xgb.DMatrix(X_tr, label=y_train, feature_names=list(feature_names))
dval = xgb.DMatrix(X_va, label=y_val, feature_names=list(feature_names))
watch = [(dtrain, 'train'), (dval, 'val')]
base = {'max_depth': 6, 'min_child_weight': 1, 'objective': 'reg:squarederror', 'nthread': 8, 'seed': 1, 'verbosity': 0}
for eta in [0.3, 0.1]:
    m = xgb.train({**base, 'eta': eta}, dtrain, num_boost_round=100, evals=watch, verbose_eval=False)
    print(f'eta={eta} RMSE={rmse(y_val, m.predict(dval)):.4f}')

eta=0.3 RMSE=1.8265
eta=0.1 RMSE=1.7248
